# Initial Cleaning

This section handles cleaning the raw data from the review and metadata files and joining them together.
I then export this and comment everything out and use the newly exported dataset as the original source of truth.

### Importing

In [140]:
import pandas as pd
import gzip
import json

def parse(path):
  g = gzip.open(path, 'rb')
  for l in g:
    yield json.loads(l)

def getDF(path):
  i = 0
  df = {}
  for d in parse(path):
    df[i] = d
    i += 1
    if i > 100000:
      break
  return pd.DataFrame.from_dict(df, orient='index')

df_reviews = getDF('data/Books_5.json.gz')
df_metadata = getDF('data/meta_Books.json.gz')

### Renaming

In [141]:
review_column_rename_map = {
    "overall": "user_rating",
    "verified": "user_verified_purchase",
    "reviewTime": "user_review_date_raw",
    "reviewerID": "user_id",
    "asin": "book_id",
    "style": "book_format",
    "reviewerName": "user_name",
    "reviewText": "user_review_text",
    "summary": "user_review_summary",
    "unixReviewTime": "user_review_timestamp",
    "vote": "user_review_helpful_votes",
}

metadata_column_rename_map = {
    "category": "book_category",
    "description": "book_description",
    "title": "book_title",
    "brand": "book_brand",
    "rank": "book_rank",
    "price": "book_price",
    "asin": "book_id",
    "imageURL": "book_image_url",
}

df_reviews = df_reviews.rename(columns=review_column_rename_map)
df_metadata = df_metadata.rename(columns=metadata_column_rename_map)

### Dropping rows and columns

In [142]:

review_columns_to_drop = [
    # "user_review_helpful_votes",
    "user_review_date_raw",
    "user_verified_purchase",
    "user_name",
    "user_review_helpful_votes",
    "image"
]
df_reviews.drop(columns=review_columns_to_drop, inplace=True)

metadata_columns_to_drop = [
    "tech1", 
    "fit", 
    "tech2", 
    "feature",  
    "also_buy", 
    "main_cat", 
    "similar_item", 
    "date",
    'imageURLHighRes',
    'also_view'
]
df_metadata.drop(columns=metadata_columns_to_drop, inplace=True)

In [143]:
df_reviews.head(5)

,user_rating,user_id,book_id,book_format,user_review_text,user_review_summary,user_review_timestamp
0,5.0,A1REUF3A1YCPHM,0001713353,{'Format:': ' Hardcover'},"The King, the Mice and the Cheese by Nancy Gur...",A story children will love and learn from,1112140800
1,5.0,AVP0HXC9FG790,0001713353,NaN,The kids loved it!,Five Stars,1466380800
2,5.0,A324TTUBKTN73A,0001713353,{'Format:': ' Paperback'},My students (3 & 4 year olds) loved this book!...,Five Stars,1453593600
3,5.0,A2RE7WG349NV5D,0001713353,{'Format:': ' Paperback'},LOVE IT,Five Stars,1436400000
4,5.0,A32B7QIUDQCD0E,0001713353,NaN,Great!,Five Stars,1421539200


### Cleaning

In [144]:
from datetime import datetime
import re

def clean_book_format(text):
    """
        Input: {'Format:': ' Paperback'} 
        Output: 'paperback'
    """
    if isinstance(text, dict):
        value = str(list(text.values())[0]).strip().lower()
        return value
    return None

def convert_unix_timestamp_to_year(timestamp):
    """
        Input: 1112140800 (March 30 2005)
        Output: 2005
    """
    year = datetime.fromtimestamp(timestamp).year
    return year

def parse_rank_from(book_rank):
    """
        Input: '1,349,781 in Books ('
        Output: 1349781
    """
    if book_rank is None or isinstance(book_rank, list):
        book_rank = "" if book_rank is None else " ".join(map(str, book_rank))
    match = re.search(r'\d[\d,]*', str(book_rank))
    if not match:
        return None
    return int(match.group().replace(',', ''))

def remove_dollar_sign(text):
    return text.strip("$")

df_reviews["book_format"] = df_reviews["book_format"].apply(clean_book_format)
df_reviews["user_review_timestamp"] = df_reviews["user_review_timestamp"].apply(convert_unix_timestamp_to_year)
df_metadata["book_price"] = df_metadata["book_price"].apply(remove_dollar_sign)
df_metadata["book_rank"] = df_metadata["book_rank"].apply(parse_rank_from)

### Preprocessing

In [145]:
from sklearn.preprocessing import StandardScaler
import torch

df_metadata["book_rank_scaled"] = StandardScaler().fit_transform(df_metadata[["book_rank"]])
book_rank_scaled_idx = torch.tensor(df_metadata["book_rank_scaled"].values, dtype=torch.float32)


In [146]:
# df = df_reviews.merge(df_metadata, on="book_id")
# df.dropna(axis=0, inplace=True)

# df.to_csv("./data/dataset.csv", index=False)

# import pandas as pd

# df = pd.read_csv("./data/dataset.csv")
# df.head()

In [147]:
import np

user_ids = df_reviews['user_id'].unique()
book_ids = df_metadata['book_id'].unique()

user2idx = {u: i for i, u in enumerate(user_ids)}
book2idx = {b: i for i, b in enumerate(book_ids)}

df_reviews["user_idx"] = df_reviews["user_id"].map(user2idx)
df_reviews["book_idx"] = df_reviews["book_id"].map(book2idx)

# Drop reviews whose book_id isn't in the metadata (metadata only covers first 100k rows)
df_reviews = df_reviews.dropna(subset=["book_idx"])
df_reviews["book_idx"] = df_reviews["book_idx"].astype(int)

df_metadata["book_idx"] = df_metadata["book_id"].map(book2idx)
df_metadata = df_metadata.dropna(subset=["book_idx"]).set_index("book_idx").sort_index()

n_users, n_books = len(user2idx), len(book2idx)

user_pos_books = df_reviews.groupby("user_idx")["book_idx"].apply(set).to_dict()
all_book_idxs = np.arange(n_books)

In [150]:
import numpy as np
import torch
from torch.utils.data import Dataset, DataLoader

class TwoTowersDataset(Dataset):
    def __init__(self, df_reviews, user_pos_books, n_books, n_users, neg_ratio=4):
        self.users = df_reviews["user_idx"].values 
        self.books = df_reviews["book_idx"].values
        self.df_reviews = df_reviews
        self.user_pos_books = user_pos_books
        self.n_books = n_books
        self.n_users = n_users
        self.neg_ratio = neg_ratio

    def __len__(self):
        return len(self.users) * (1 + self.neg_ratio) 
        
    def __getitem__(self, idx):
        pos_idx = idx // (1 + self.neg_ratio)
        is_positive = (idx % (1 + self.neg_ratio)) == 0

        user_idx = self.users[pos_idx]

        if is_positive:
            book_idx = self.books[pos_idx]                      # positive interaction
            label = 1.0
        else:
            book_idx = np.random.randint(self.n_books)
            while book_idx in self.user_pos_books[user_idx]:    # reject books they've read
                book_idx = np.random.randint(self.n_books)
            label = -1.0

        return {
            "user_idx": torch.tensor(user_idx, dtype=torch.long),
            "book_rank_scaled": book_rank_scaled_idx[book_idx],
            "book_idx": torch.tensor(book_idx, dtype=torch.long),
            "label": torch.tensor(label, dtype=torch.float32),
        }


In [151]:
dataset = TwoTowersDataset(
    df_reviews,
    user_pos_books,
    n_books,
    n_users
)

dataset[2]

{'user_idx': tensor(0),
 'book_rank_scaled': tensor(-0.8870),
 'book_idx': tensor(94551),
 'label': tensor(-1.)}